# 5 · Shared evaluation - self-consistency (AlphaFold2, single-sequence)
Run AFTER the design notebooks. Upload every `designs_<model>.csv` (or the whole
`outputs/` folder) plus the bundle.

**One identical refolder for every model** = comparable realism. We use **AF2
(ColabFold) in single-sequence mode**, because:
1. the input templates are AF2 v6 structures, so refolding with AF2 compares
   like-to-like (same predictor's geometry for reference and refold);
2. Caliby's built-in self-consistency already uses AF2 - this matches it;
3. AF2 is the field-standard self-consistency oracle.

**Single-sequence** (`--msa-mode single_sequence`) is essential: designed
sequences have no meaningful MSA, and building one would leak WT-homolog
information and inflate the score. Settings match Caliby: 5 models, 3 recycles.

*(ESMFold is a faster fallback - see the optional cell - but AF2 single-seq is
the default for consistency with the rest of the pipeline.)* **Runtime → GPU.**

In [ ]:
#@title Step 0 - Upload & unzip the design bundle
#@markdown Upload **design_bundle.zip** (contains `design_common.py`,
#@markdown `design_input_proteins.csv`, and `structures/`).
#@markdown Build it locally with `design/make_bundle.sh`.
import os, zipfile
from google.colab import files

if not os.path.exists("design_common.py"):
    print("Upload design_bundle.zip:")
    up = files.upload()
    zname = next(iter(up))
    with zipfile.ZipFile(zname) as z:
        z.extractall(".")
    # if it unzipped into a 'design/' subdir, hoist contents to CWD
    if os.path.exists("design/design_common.py") and not os.path.exists("design_common.py"):
        import shutil
        for item in os.listdir("design"):
            shutil.move(os.path.join("design", item), item)
print("Bundle ready:", sorted(os.listdir(".")))

In [ ]:
#@title Install ColabFold (AF2) + TM-align
import subprocess, sys
# ColabFold gives AF2 weights + a batch CLI; no MMseqs server needed in single-sequence mode.
subprocess.run([sys.executable,"-m","pip","install","--quiet",
    "colabfold[alphafold] @ git+https://github.com/sokrypton/ColabFold",
    "tmtools","biotite"], check=False)
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (set GPU runtime!)")
print("AF2 weights download on first prediction (~3-4 GB).")

In [ ]:
#@title Step 1 - Collect all designs_<model>.csv and write ONE FASTA
import design_common as dc, pandas as pd, glob, os
from google.colab import files
csvs = glob.glob(str(dc.OUTPUT_DIR / "designs_*.csv"))
if not csvs:
    print("No designs found in outputs/. Upload them:")
    up = files.upload()
    for fn in up:
        if fn.endswith(".csv"):
            os.makedirs(dc.OUTPUT_DIR, exist_ok=True)
            os.rename(fn, str(dc.OUTPUT_DIR / os.path.basename(fn)))
    csvs = glob.glob(str(dc.OUTPUT_DIR / "designs_*.csv"))
alld = pd.concat([pd.read_csv(c) for c in csvs], ignore_index=True)
alld["design_id"] = (alld["uniprot_id"] + "__" + alld["model"].str.replace("/","-")
                     + "__s" + alld["sample_idx"].astype(str))
print(f"{len(alld)} designs from models: {sorted(alld['model'].unique())}")
display(alld.groupby("model").size().rename("n_designs"))

FASTA = dc.OUTPUT_DIR / "all_designs.fasta"
with open(FASTA, "w") as fh:
    for r in alld.itertuples():
        fh.write(f">{r.design_id}\n{r.designed_sequence}\n")
print("wrote", FASTA)

In [ ]:
#@title Step 2 - Fold every design with AF2 single-sequence (5 models, 3 recycles)
#@markdown Matches Caliby's self-consistency settings. This is the slow step
#@markdown (~thousands of short folds); leave it running.
import subprocess, sys
AF2_DIR = str(dc.OUTPUT_DIR / "af2_sc")
cmd = [sys.executable, "-m", "colabfold.batch",
       str(dc.OUTPUT_DIR / "all_designs.fasta"), AF2_DIR,
       "--msa-mode", "single_sequence",
       "--num-models", str(dc.CONFIG.sc_num_models),
       "--num-recycle", str(dc.CONFIG.sc_num_recycles),
       "--model-type", "alphafold2_ptm"]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
print("AF2 predictions written to", AF2_DIR)

In [ ]:
#@title Step 3 - Self-consistency: AF2 refold vs input backbone (TM-score, RMSD, pLDDT)
import design_common as dc, numpy as np, pandas as pd, glob, os, json, re
from tqdm.auto import tqdm
from tmtools import tm_align
from tmtools.io import get_structure, get_residue_data

proteins = dc.load_inputs()
ref_path = dict(zip(proteins.uniprot_id, proteins.structure_path))
AF2_DIR = str(dc.OUTPUT_DIR / "af2_sc")

def ca_coords_seq(pdb_path):
    chain = next(get_structure(pdb_path).get_chains())
    return get_residue_data(chain)            # (coords, seq)

def best_pred_pdb(design_id):
    # ColabFold writes <id>_unrelaxed_rank_001_..._model_*.pdb ; take rank_001
    hits = sorted(glob.glob(os.path.join(AF2_DIR, f"{design_id}_*rank_001*.pdb")))
    return hits[0] if hits else None

def mean_plddt(pdb_path):
    vals = [float(l[60:66]) for l in open(pdb_path)
            if l.startswith("ATOM") and l[12:16].strip()=="CA"]
    return sum(vals)/len(vals) if vals else float("nan")

records = []
for r in tqdm(list(alld.itertuples()), desc="score AF2 refolds"):
    pred = best_pred_pdb(r.design_id)
    if pred is None:
        records.append({"design_id":r.design_id, "uniprot_id":r.uniprot_id,
                        "model":r.model, "sample_idx":r.sample_idx,
                        "sc_tm_score":np.nan,"sc_rmsd":np.nan,"refold_plddt":np.nan})
        continue
    cp, sp = ca_coords_seq(pred)
    cr, sr = ca_coords_seq(ref_path[r.uniprot_id])
    res = tm_align(cp, cr, sp, sr)
    records.append({"design_id":r.design_id, "uniprot_id":r.uniprot_id,
                    "model":r.model, "sample_idx":r.sample_idx,
                    "sc_tm_score":res.tm_norm_chain2, "sc_rmsd":res.rmsd,
                    "refold_plddt":mean_plddt(pred)})
sc = pd.DataFrame(records).merge(
    alld[["design_id","domain","rank_class","target_cell","model_score"]],
    on="design_id", how="left")
out = dc.OUTPUT_DIR / "self_consistency_all_models.csv"
sc.to_csv(out, index=False)
print("wrote", out, f"({sc['sc_tm_score'].notna().sum()} scored)")
display(sc.groupby("model")[["sc_tm_score","sc_rmsd","refold_plddt"]].median().round(3))

### Optional - ESMFold fallback
If ColabFold install is flaky on a given runtime, ESMFold is a faster
single-sequence alternative. **Use only if you refold *every* model's designs
with it** (never mix refolders within one analysis). It is a *different*
predictor from the AF2 v6 templates, so TM-scores shift slightly vs the AF2 path.

In [ ]:
#@title (optional) ESMFold fallback refolder - only if AF2/ColabFold failed
RUN_ESMFOLD_FALLBACK = False  #@param {type:"boolean"}
if RUN_ESMFOLD_FALLBACK:
    import subprocess, sys, torch, numpy as np, pandas as pd, tempfile
    from tqdm.auto import tqdm
    from tmtools import tm_align
    from tmtools.io import get_structure, get_residue_data
    subprocess.run([sys.executable,"-m","pip","install","--quiet","fair-esm[esmfold]"], check=False)
    import esm
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    fold = esm.pretrained.esmfold_v1().eval().to(device); fold.set_chunk_size(128)
    proteins = dc.load_inputs(); ref_path = dict(zip(proteins.uniprot_id, proteins.structure_path))
    def ca(pdb_path):
        return get_residue_data(next(get_structure(pdb_path).get_chains()))
    @torch.no_grad()
    def refold(seq):
        out = fold.infer_pdb(seq)
        with tempfile.NamedTemporaryFile("w",suffix=".pdb",delete=False) as f:
            f.write(out); path=f.name
        pl=[float(l[60:66]) for l in out.splitlines() if l.startswith("ATOM") and l[12:16].strip()=="CA"]
        return path, (sum(pl)/len(pl) if pl else float("nan"))
    rec=[]
    for r in tqdm(list(alld.itertuples()), desc="ESMFold refold"):
        pp, pl = refold(r.designed_sequence)
        cp,sp = ca(pp); cr,sr = ca(ref_path[r.uniprot_id])
        res = tm_align(cp,cr,sp,sr)
        rec.append({"design_id":r.design_id,"uniprot_id":r.uniprot_id,"model":r.model,
                    "sample_idx":r.sample_idx,"sc_tm_score":res.tm_norm_chain2,
                    "sc_rmsd":res.rmsd,"refold_plddt":pl})
    sc_esm = pd.DataFrame(rec)
    sc_esm.to_csv(dc.OUTPUT_DIR / "self_consistency_all_models_ESMFOLD.csv", index=False)
    display(sc_esm.groupby("model")[["sc_tm_score","sc_rmsd","refold_plddt"]].median().round(3))

### Output
`self_consistency_all_models.csv` - one row per design with `sc_tm_score`,
`sc_rmsd`, `refold_plddt` (all from AF2 single-sequence), plus `domain`,
`rank_class`, `target_cell`, and the model's own `model_score`. Feed this and the
biophysical-property columns from your WT-shift pipeline into the Stage-3 PCA /
Cohen's-d analysis. Identical design config + identical refolder ⇒ differences
are attributable to the model, not the protocol.